In [ ]:
import pandas as pd
import numpy as np
import os
import re
import glob
from zoneinfo import ZoneInfo
from datetime import datetime
from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

dir_path = "D:/research assistant/EEG/Machine Learning/"
log_path = os.path.join(dir_path, "log_VBL")
processed_path = os.path.join(dir_path, "Processed VBL Files")
segment_path = os.path.join(dir_path, "segment_VBL")
eeg_file = os.path.join(dir_path, "all_subjects_power_bands_averaged_channels.xlsx")
label_dir = os.path.join(dir_path, "labelled files_VBL")

In [2]:
action_categ = {"Instructions": "switch_page",
                "read instruction": "switch_page",
            	"Video 1": "switch_page",
                "notes": "type",
                "plan": "type",
            	"Video 2": "switch_page",
                "03_q1a": "switch_page",
            	"03_q1b": "switch_page",
            	"03_q1c": "switch_page",
            	"03_q2a": "switch_page",
                "03_q2b": "switch_page",
                "03_q2c": "switch_page",
                "03_q3a": "switch_page",
            	"03_q3b": "switch_page",
            	"03_q3c": "switch_page",
            	"03_q4a": "switch_page",
            	"03_q4b": "switch_page",
                "03_q4c": "switch_page",
                "03_q5a": "switch_page",
                "03_q5b": "switch_page",
                "03_q5c": "switch_page",
            	"04_q1a": "switch_page",
                "04_q1b": "switch_page",
                "04_q2a": "switch_page",
                "04_q2b": "switch_page",
            	"04_q3a": "switch_page",
                "04_q3b": "switch_page",
                "04_q4a": "switch_page",
                "04_q4b": "switch_page",
                "04_q5a": "switch_page",
                "04_q5b": "switch_page"
}

In [3]:
def get_file(pattern, path):
    matching_files = glob.glob(pattern)
    if not matching_files:
        raise FileNotFoundError(f"No file found for subject with pattern: {pattern}")
    file = os.path.join(path, matching_files[0])
    df = pd.read_excel(file)
    return df

In [4]:
columns = ['play_video', 'pause_video', 'switch_page', 'quiz', 'type', 'time_checking', 'content_reading']
action_df = pd.DataFrame()
for filename in os.listdir(processed_path):
    subject_data = pd.DataFrame()
    processed_file_path = os.path.join(processed_path, filename)
    # read processed files, we only need the time segment so any sheet will do
    processed_df = pd.read_excel(processed_file_path, sheet_name="process")
    # find the index of the subject to locate the related clean file and raw file
    number = re.findall(r'p0*(\d*)', filename)[0]
    try:
        # get the corresponding clean file
        log_pattern = os.path.join(log_path, f"log-*_subject{number}.xlsx")
        log_df = get_file(log_pattern, log_path)
        log_df['time'] = pd.to_datetime(log_df['time'], format="%Y-%m-%d %H:%M:%S").dt.tz_localize("Asia/Singapore").astype("int64") // 10**9
        log_df['action'] = log_df['action'].replace(action_categ)

        # get the corresponding labelled file
        label_pattern = os.path.join(label_dir, f"p{number.zfill(3)}*.xlsx")
        label_df = get_file(label_pattern, label_dir)
        label_df['time'] = pd.to_datetime(label_df['time'], format="%Y-%m-%d %H:%M:%S").dt.tz_localize("Asia/Singapore").astype("int64") // 10**9
        check_time_col = []
        read_content_col = []
        label_df = label_df.fillna("")
        label_df.columns = label_df.columns.str.strip()
        if "validation" in label_df.columns:
            label_df = label_df.rename(columns={"validation": "Validation"})
        for i, r in label_df.iterrows():
            if r['validation_checked']:
                check_time_col.append("time_checking") if "time_checking" in r['validation_checked'] else check_time_col.append("")
            elif r['Validation']:
                check_time_col.append("time_checking") if "time_checking" in r['Validation'] else check_time_col.append("")
            elif r['label']:
                check_time_col.append("time_checking") if "time_checking" in r['label'] else check_time_col.append("")
            else:
                check_time_col.append("")
            
            if r['validation_checked']:
                read_content_col.append("content_reading") if "content_reading" in r['validation_checked'] else read_content_col.append("")
            elif r['Validation']:
                read_content_col.append("content_reading") if "content_reading" in r['Validation'] else read_content_col.append("")
            elif r['label']:        
                read_content_col.append("content_reading") if "content_reading" in r['label'] else read_content_col.append("")
            else:
                read_content_col.append("")
        label_df['time_checking'] = check_time_col
        label_df['content_reading'] = read_content_col
        label_df = label_df[['time', 'time_checking', 'content_reading']]
        df = pd.merge(label_df, log_df, on="time", how='left').fillna("")
    except Exception as e:
        print(number)
        print(e)
    output_path = os.path.join(segment_path, number)
    filename_col = []
    for index, row in processed_df.iterrows():
        # convert the epoch time to local time zone
        start = row['start_time'].replace(tzinfo=ZoneInfo("Asia/Singapore")).timestamp()
        end = row['end_time'].replace(tzinfo=ZoneInfo("Asia/Singapore")).timestamp()
        start_time= row['start_time'].strftime("%Y-%m-%d %H-%M-%S")
        end_time = row['end_time'].strftime("%Y-%m-%d %H-%M-%S")
        filename_col.append(f"{start_time}_{end_time}.xlsx")
        segment = df[(df['time'] >= start) & (df['time'] < end + 1)]
        action_counts = segment['action'].value_counts()
        label_segment = segment[['time', 'time_checking', 'content_reading']].drop_duplicates(subset=['time'])
        label_counts = {}
        for col in ['time_checking', 'content_reading']:
            vc = label_segment[col].value_counts()
            for value, count in vc.items():
                if value != '':
                    label_counts[value] = count
        counts = action_counts.to_dict()
        counts.update(label_counts)
        segment_row = pd.DataFrame([counts])
        subject_data = pd.concat([subject_data, segment_row], ignore_index=True).fillna(0).astype(int)
    subject_data.columns = [
        "".join(map(str, col)) if isinstance(col, tuple) else str(col)
        for col in subject_data.columns
    ]
    subject_data = subject_data.reindex(columns=columns)
    subject_data = subject_data.fillna(0)
    subject_data['subject'] = int(number)
    subject_data['segment_file'] = filename_col
    action_df = pd.concat([action_df, subject_data], ignore_index=True)

In [5]:
eeg_df = pd.read_excel(eeg_file)
final_df = pd.merge(action_df, eeg_df, on=["segment_file", "subject"])
final_df = final_df[final_df["process_category"] != "OT"]
final_df.to_excel(os.path.join(dir_path, "combined.xlsx"))
cols = ["Delta", "Theta", "Alpha", "Beta", "Gamma", 'play_video', 'pause_video', 'switch_page', 'quiz', 'type', 'time_checking', 'content_reading']
scaler = StandardScaler()
final_df[cols] = scaler.fit_transform(final_df[cols])
final_df = final_df.fillna(0)

In [9]:
final_df['process_category'].count()

10152

In [23]:
# Machine Learning
target_col = "process_category"
# action + eeg columns + content_reading
X = final_df[['play_video', 'pause_video', 'switch_page', 'quiz', 'type', "Delta", "Theta",	"Alpha", "Beta", "Gamma", "content_reading"]]
y = final_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
automl = AutoML()
automl.fit(
    X_train=X_train,
    y_train=y_train,
    task="classification",   # or "regression"
    time_budget=120,
    metric="accuracy"
)
print("Best model:", automl.model)
print("Best config:", automl.best_config)

y_pred = automl.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm, index=automl.classes_, columns=automl.classes_))

[flaml.automl.logger: 04-13 11:40:58] {2375} INFO - task = classification
[flaml.automl.logger: 04-13 11:40:58] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 04-13 11:40:58] {2489} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 04-13 11:40:58] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 04-13 11:40:58] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 04-13 11:40:59] {3046} INFO - Estimated sufficient time budget=863s. Estimated necessary time budget=20s.
[flaml.automl.logger: 04-13 11:40:59] {3097} INFO -  at 0.1s,	estimator lgbm's best error=3.1818e-01,	best estimator lgbm's best error=3.1818e-01
[flaml.automl.logger: 04-13 11:40:59] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 04-13 11:40:59] {3097} INFO -  at 0.2s,	estimator lgbm's best error=3.1818e-01,	best estimator lgbm's best error=3.1818e-01
[flaml.autom

In [16]:
target_col = "process_category"
# action + eeg columns + time_checking
X = final_df[['play_video', 'pause_video', 'switch_page', 'quiz', 'type', "Delta", "Theta",	"Alpha", "Beta", "Gamma", "time_checking"]]
y = final_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
automl = AutoML()
automl.fit(
    X_train=X_train,
    y_train=y_train,
    task="classification",   # or "regression"
    time_budget=60,
    metric="accuracy"
)
print("Best model:", automl.model)
print("Best config:", automl.best_config)

y_pred = automl.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm, index=automl.classes_, columns=automl.classes_))

[flaml.automl.logger: 04-13 10:38:42] {2375} INFO - task = classification
[flaml.automl.logger: 04-13 10:38:42] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 04-13 10:38:42] {2489} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 04-13 10:38:42] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 04-13 10:38:42] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 04-13 10:38:42] {3046} INFO - Estimated sufficient time budget=815s. Estimated necessary time budget=19s.
[flaml.automl.logger: 04-13 10:38:42] {3097} INFO -  at 0.1s,	estimator lgbm's best error=3.1959e-01,	best estimator lgbm's best error=3.1959e-01
[flaml.automl.logger: 04-13 10:38:42] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 04-13 10:38:42] {3097} INFO -  at 0.2s,	estimator lgbm's best error=3.1959e-01,	best estimator lgbm's best error=3.1959e-01
[flaml.autom

In [17]:
target_col = "process_category"
# action + eeg columns + content_reading
X = final_df[['play_video', 'pause_video', 'switch_page', 'quiz', 'type', "Delta", "Theta",	"Alpha", "Beta", "Gamma", "time_checking", "content_reading"]]
y = final_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
automl = AutoML()
automl.fit(
    X_train=X_train,
    y_train=y_train,
    task="classification",   # or "regression"
    time_budget=60,
    metric="accuracy"
)
print("Best model:", automl.model)
print("Best config:", automl.best_config)

y_pred = automl.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm, index=automl.classes_, columns=automl.classes_))

[flaml.automl.logger: 04-13 10:39:44] {2375} INFO - task = classification
[flaml.automl.logger: 04-13 10:39:44] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 04-13 10:39:44] {2489} INFO - Minimizing error metric: 1-accuracy
[flaml.automl.logger: 04-13 10:39:44] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd', 'lrl1']
[flaml.automl.logger: 04-13 10:39:44] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 04-13 10:39:44] {3046} INFO - Estimated sufficient time budget=915s. Estimated necessary time budget=21s.
[flaml.automl.logger: 04-13 10:39:44] {3097} INFO -  at 0.1s,	estimator lgbm's best error=2.7428e-01,	best estimator lgbm's best error=2.7428e-01
[flaml.automl.logger: 04-13 10:39:44] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 04-13 10:39:44] {3097} INFO -  at 0.2s,	estimator lgbm's best error=2.7428e-01,	best estimator lgbm's best error=2.7428e-01
[flaml.autom